# Stage 2 — Hyperbolic Ops Module (Poincaré Ball)

Standalone math utilities for the Poincaré ball model of hyperbolic space.
No dependency on Stage 1's graph or any dataset -- this is pure geometry that
Stage 3 (RotH) will import and build the entity/relation embeddings on top of.

Curvature convention used throughout: `c > 0` is the *magnitude* of curvature,
and the ball has constant curvature `-c` (so `c -> 0` recovers flat/Euclidean
space). This matches the RotH paper's convention and is what lets Stage 3 learn
a per-relation curvature via a log-parameterization (`c = softplus(log_c)`,
guaranteeing `c > 0` while allowing gradient descent on an unconstrained
parameter).

Functions:
- `project_to_ball` — clip a point back inside the ball of radius `1/sqrt(c)`
- `mobius_add` — hyperbolic (Möbius) addition, the ball's analogue of `x + y`
- `exp_map_zero` — tangent space at the origin → the ball
- `log_map_zero` — the ball → tangent space at the origin
- `geodesic_distance` — hyperbolic distance between two points on the ball
- `givens_rotation` — paired 2D rotations over embedding dimensions (RotH's relation transform)


## 0. Setup

In [5]:
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=6, sci_mode=False)


## 1. Numerical-stability constants

The two danger zones in Poincaré-ball math are:
- **The ball boundary** (`||x|| -> 1/sqrt(c)`): `artanh` blows up as its argument
  approaches 1, and several formulas have `1 - c||x||^2` in a denominator, which
  goes to 0 at the boundary.
- **The origin** (`||x|| -> 0`): normalizing by `||x||` (as in `exp_map_zero` /
  `log_map_zero`) divides by ~0 for points sitting exactly at the origin.

`BALL_EPS` keeps points strictly inside the ball; `MIN_NORM` guards divisions by
a near-zero norm.


In [6]:
BALL_EPS = 1e-5   # keep norms at most (1 - BALL_EPS) / sqrt(c)
MIN_NORM = 1e-15  # floor for norms used as denominators


## 2. `project_to_ball`

Clips a point's norm so it stays strictly inside the ball of radius `1/sqrt(c)`.
Call this after every optimizer step in Stage 4 (`project_entities()`), since
plain Euclidean Adam updates can walk a point outside the ball.


In [7]:
def project_to_ball(x: torch.Tensor, c: torch.Tensor, eps: float = BALL_EPS) -> torch.Tensor:
    """Project x back onto the open Poincare ball of curvature -c (radius 1/sqrt(c)).

    x: (..., dim) tensor of points (already in the ball, or just off it after an update)
    c: scalar or (...,) broadcastable tensor of positive curvatures
    """
    c = c.clamp_min(MIN_NORM)
    max_norm = (1.0 - eps) / c.sqrt()
    norm = x.norm(dim=-1, keepdim=True).clamp_min(MIN_NORM)
    factor = torch.where(norm > max_norm, max_norm / norm, torch.ones_like(norm))
    return x * factor


## 3. `mobius_add`

Hyperbolic addition on the ball:

```
x (+)_c y = ((1 + 2c<x,y> + c||y||^2) x + (1 - c||x||^2) y)
            -----------------------------------------------
                  1 + 2c<x,y> + c^2 ||x||^2 ||y||^2
```

Reduces to ordinary vector addition as `c -> 0`.


In [8]:
def mobius_add(x: torch.Tensor, y: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Mobius addition x (+)_c y on the Poincare ball of curvature -c."""
    x2 = (x * x).sum(dim=-1, keepdim=True)
    y2 = (y * y).sum(dim=-1, keepdim=True)
    xy = (x * y).sum(dim=-1, keepdim=True)

    num = (1 + 2 * c * xy + c * y2) * x + (1 - c * x2) * y
    denom = 1 + 2 * c * xy + (c ** 2) * x2 * y2
    return num / denom.clamp_min(MIN_NORM)


## 4. `exp_map_zero` / `log_map_zero`

The exponential map at the origin sends a tangent vector to a point on the
ball; the logarithmic map at the origin is its inverse. These are the maps
Stage 6 uses to move POI embeddings between hyperbolic space and LLaDA's
Euclidean embedding space (`log_map_zero` is the one that was defined-but-unused
in the reference notebook).

```
exp_0^c(v) = tanh(sqrt(c)||v||) * v / (sqrt(c)||v||)
log_0^c(y) = artanh(sqrt(c)||y||) * y / (sqrt(c)||y||)
```


In [9]:
def exp_map_zero(v: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Exponential map at the origin: tangent space -> Poincare ball."""
    sqrt_c = c.clamp_min(MIN_NORM).sqrt()
    v_norm = v.norm(dim=-1, keepdim=True).clamp_min(MIN_NORM)
    coeff = torch.tanh(sqrt_c * v_norm) / (sqrt_c * v_norm)
    return project_to_ball(coeff * v, c)


def log_map_zero(y: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Logarithmic map at the origin: Poincare ball -> tangent space (inverse of exp_map_zero)."""
    sqrt_c = c.clamp_min(MIN_NORM).sqrt()
    y_norm = y.norm(dim=-1, keepdim=True).clamp(MIN_NORM, (1.0 - BALL_EPS) / sqrt_c)
    coeff = torch.atanh(sqrt_c * y_norm) / (sqrt_c * y_norm)
    return coeff * y


## 5. `geodesic_distance`

Hyperbolic distance between two points on the ball, via Mobius subtraction:

```
d_c(x, y) = (2 / sqrt(c)) * artanh( sqrt(c) * || (-x) (+)_c y || )
```

This is the function RotH's scoring function is built on (`score = -d_c(x, y)^2 + biases`).


In [10]:
def geodesic_distance(x: torch.Tensor, y: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """Hyperbolic (geodesic) distance between x and y on the Poincare ball of curvature -c.
    Returns a tensor with the last dim reduced (shape x.shape[:-1])."""
    sqrt_c = c.clamp_min(MIN_NORM).sqrt()
    diff = mobius_add(-x, y, c)
    diff_norm = diff.norm(dim=-1, keepdim=True).clamp(MIN_NORM, (1.0 - BALL_EPS) / sqrt_c)
    dist = 2.0 * torch.atanh(sqrt_c * diff_norm) / sqrt_c
    return dist.squeeze(-1)


## 6. `givens_rotation`

RotH applies a per-relation rotation in *tangent space* before translating and
mapping back onto the ball. The rotation is a product of 2D Givens rotations
over consecutive dimension pairs `(0,1), (2,3), ...` -- the hyperbolic analogue
of RotatE's complex-plane rotation. `dim` must be even.


In [11]:
def givens_rotation(x: torch.Tensor, theta: torch.Tensor) -> torch.Tensor:
    """Apply per-pair 2D Givens rotations to the last dimension of x.

    x:     (..., dim), dim must be even
    theta: (..., dim/2) rotation angles, one per (2i, 2i+1) coordinate pair,
           broadcastable against x's leading dims
    """
    *lead, dim = x.shape
    assert dim % 2 == 0, "givens_rotation requires an even embedding dimension"

    x_pairs = x.view(*lead, dim // 2, 2)
    x_even, x_odd = x_pairs[..., 0], x_pairs[..., 1]

    cos_t, sin_t = torch.cos(theta), torch.sin(theta)
    rot_even = cos_t * x_even - sin_t * x_odd
    rot_odd = sin_t * x_even + cos_t * x_odd

    return torch.stack([rot_even, rot_odd], dim=-1).view(*lead, dim)


## 7. Curvature parameterization helper

Stage 3 learns one raw scalar per relation (`log_c`) and maps it through
`softplus` to guarantee `c > 0` everywhere in the optimization, without a hard
clamp that would zero out gradients at the boundary.


In [12]:
def curvature_from_logit(log_c: torch.Tensor, min_c: float = 1e-4) -> torch.Tensor:
    """Map an unconstrained learnable scalar to a strictly positive curvature."""
    return F.softplus(log_c) + min_c


## 8. Self-checks

None of these depend on Stage 1/3 -- just verifying the geometry is internally
consistent before RotH gets built on top of it. Uses `float64` for the checks
(tighter tolerances than the `float32` training will actually run in).


In [13]:
torch.manual_seed(0)
DIM = 8
BATCH = 5
dtype = torch.float64

c = torch.tensor(0.7, dtype=dtype)

def random_ball_points(n, dim, c, dtype, max_norm_frac=0.7):
    v = torch.randn(n, dim, dtype=dtype)
    v = v / v.norm(dim=-1, keepdim=True)
    r = torch.rand(n, 1, dtype=dtype) * max_norm_frac / c.sqrt()
    return v * r

x = random_ball_points(BATCH, DIM, c, dtype)
y = random_ball_points(BATCH, DIM, c, dtype)


In [14]:
# --- project_to_ball keeps points strictly inside the ball -----------------
far_points = torch.randn(BATCH, DIM, dtype=dtype) * 10.0  # deliberately outside
projected = project_to_ball(far_points, c)
max_allowed = (1.0 - BALL_EPS) / c.sqrt()
norms = projected.norm(dim=-1)
ok = bool((norms <= max_allowed + 1e-9).all())
print(f"[project_to_ball] all norms <= 1/sqrt(c) boundary: {ok}  "
      f"(max norm={norms.max().item():.6f}, boundary={max_allowed.item():.6f})")
assert ok


[project_to_ball] all norms <= 1/sqrt(c) boundary: True  (max norm=1.195217, boundary=1.195217)


In [15]:
# --- mobius_add(0, y) == y --------------------------------------------------
zero = torch.zeros_like(y)
out = mobius_add(zero, y, c)
err = (out - y).abs().max().item()
print(f"[mobius_add] mobius_add(0, y) == y, max abs error: {err:.2e}")
assert err < 1e-8


[mobius_add] mobius_add(0, y) == y, max abs error: 0.00e+00


In [16]:
# --- mobius_add(x, -x) == 0 --------------------------------------------------
out = mobius_add(x, -x, c)
err = out.abs().max().item()
print(f"[mobius_add] mobius_add(x, -x) == 0, max abs error: {err:.2e}")
assert err < 1e-8


[mobius_add] mobius_add(x, -x) == 0, max abs error: 2.17e-19


In [17]:
# --- exp_map_zero / log_map_zero are inverses on the tangent space -----------
v = torch.randn(BATCH, DIM, dtype=dtype) * 0.3  # modest tangent vectors, stay well off the boundary
y_ball = exp_map_zero(v, c)
v_recovered = log_map_zero(y_ball, c)
err = (v - v_recovered).abs().max().item()
print(f"[exp/log_map_zero] round-trip max abs error: {err:.2e}")
assert err < 1e-6


[exp/log_map_zero] round-trip max abs error: 1.11e-16


In [18]:
# --- geodesic_distance(x, x) == 0 --------------------------------------------
d_self = geodesic_distance(x, x, c)
err = d_self.abs().max().item()
print(f"[geodesic_distance] d(x, x) == 0, max abs error: {err:.2e}")
assert err < 1e-8


[geodesic_distance] d(x, x) == 0, max abs error: 2.00e-15


In [19]:
# --- geodesic_distance is symmetric ------------------------------------------
d_xy = geodesic_distance(x, y, c)
d_yx = geodesic_distance(y, x, c)
err = (d_xy - d_yx).abs().max().item()
print(f"[geodesic_distance] symmetry max abs error: {err:.2e}")
assert err < 1e-6


[geodesic_distance] symmetry max abs error: 8.88e-16


In [20]:
# --- as c -> 0, geodesic_distance -> 2x Euclidean distance -----------------------
# NOTE: the standard Poincare-ball metric has a conformal factor of 4 at the
# origin (ds^2 = 4/(1-c||x||^2)^2 ||dx||^2), so hyperbolic distance between
# points near the origin is exactly 2x their Euclidean distance, not 1x -- this
# is the standard convention (Nickel & Kiela 2017, Ganea et al. 2018), not a bug.
c_tiny = torch.tensor(1e-6, dtype=dtype)
x_small = x * 0.01  # keep points well inside a ball whose radius is huge when c is tiny
y_small = y * 0.01
d_hyp = geodesic_distance(x_small, y_small, c_tiny)
d_euc = 2.0 * (x_small - y_small).norm(dim=-1)
err = (d_hyp - d_euc).abs().max().item()
print(f"[geodesic_distance] c->0 matches 2x Euclidean distance, max abs error: {err:.2e}")
assert err < 1e-4


[geodesic_distance] c->0 matches 2x Euclidean distance, max abs error: 5.00e-13


In [21]:
# --- givens_rotation is an isometry (preserves norm) --------------------------
theta = torch.rand(BATCH, DIM // 2, dtype=dtype) * 2 * torch.pi
x_rot = givens_rotation(x, theta)
err = (x_rot.norm(dim=-1) - x.norm(dim=-1)).abs().max().item()
print(f"[givens_rotation] norm-preserving, max abs error: {err:.2e}")
assert err < 1e-10


[givens_rotation] norm-preserving, max abs error: 5.55e-17


In [22]:
# --- curvature_from_logit is always strictly positive --------------------------
logits = torch.tensor([-50.0, -1.0, 0.0, 1.0, 50.0], dtype=dtype)
c_vals = curvature_from_logit(logits)
print("curvature_from_logit(", logits.tolist(), ") ->", c_vals.tolist())
assert bool((c_vals > 0).all())


curvature_from_logit( [-50.0, -1.0, 0.0, 1.0, 50.0] ) -> [0.0001, 0.31336168751822285, 0.6932471805599453, 1.3133616875182228, 50.0001]


In [23]:
print("\nAll Stage 2 self-checks passed.")


All Stage 2 self-checks passed.
